In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Sales Data Analysis & Visualization - Tutorial\n",
    "\n",
    "This notebook walks through the whole project, step by step:\n",
    "\n",
    "1. Load a raw CSV file\n",
    "2. Look for problems in the data\n",
    "3. Clean it\n",
    "4. Calculate descriptive statistics\n",
    "5. Draw a line chart, bar charts, histograms and a correlation heatmap\n",
    "6. Export the results\n",
    "\n",
    "**Before you start**, create the sample data once. Run this in a terminal, inside the project folder:\n",
    "\n",
    "    python -m dataviz.sample_data\n",
    "\n",
    "Then run the cells below one at a time with **Shift + Enter**."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "from pathlib import Path\n",
    "\n",
    "# Make the project importable when this notebook is opened from the notebooks/ folder\n",
    "PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()\n",
    "sys.path.insert(0, str(PROJECT_ROOT))\n",
    "\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "from IPython.display import display\n",
    "\n",
    "from dataviz import config\n",
    "from dataviz.loader import load_csv, describe_load\n",
    "from dataviz.cleaning import clean_sales_data\n",
    "from dataviz.statistics import (\n",
    "    describe_numeric, describe_categorical, summarize_by,\n",
    "    revenue_by_month, revenue_by_category_region,\n",
    "    correlation_matrix, strongest_correlations, key_metrics,\n",
    ")\n",
    "from dataviz.plots.line import plot_monthly_revenue\n",
    "from dataviz.plots.bar import plot_bar, plot_grouped_bar\n",
    "from dataviz.plots.histogram import plot_histogram, plot_histogram_grid\n",
    "from dataviz.plots.heatmap import plot_correlation_heatmap\n",
    "from dataviz.export import export_clean_data, export_tables\n",
    "\n",
    "pd.set_option('display.max_columns', 20)\n",
    "config.apply_plot_style()\n",
    "print('Project folder:', PROJECT_ROOT)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 1 - Load the raw data\n",
    "\n",
    "`load_csv` reads every column as **text** on purpose. Real files contain surprises such as a dollar sign inside a number column, and reading everything as text means nothing is silently changed or lost. The cleaning step turns the text into proper numbers and dates."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "raw = load_csv()\n",
    "print(describe_load(raw))\n",
    "raw.head(10)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 2 - Look for problems\n",
    "\n",
    "Always look at raw data before you trust it. This file was made messy on purpose, so you should find missing values, duplicate orders, odd spellings and revenue written as text."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print('Missing values per column:')\n",
    "print(raw.isna().sum())\n",
    "print()\n",
    "print('Duplicate order ids:', raw['order_id'].duplicated().sum())\n",
    "print('Different spellings of category:', sorted(raw['category'].dropna().unique()))\n",
    "print('Revenue values written with a $ sign:', raw['revenue'].astype(str).str.startswith('$').sum())\n",
    "print('Customer ages above 100:', (pd.to_numeric(raw['customer_age'], errors='coerce') > 100).sum())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 3 - Clean the data\n",
    "\n",
    "`clean_sales_data` applies the cleaning steps in the right order and returns the clean table **and a report** of what it changed. The raw table is not modified."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "clean, report = clean_sales_data(raw)\n",
    "print(report.summary())\n",
    "clean.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 4 - Descriptive statistics\n",
    "\n",
    "Numbers first, charts second. Compare the **mean** with the **median**: a big gap means a few very large values pull the average up (look at the `skew` column too)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "display(describe_numeric(clean))\n",
    "display(describe_categorical(clean, 'category'))\n",
    "display(summarize_by(clean, 'region'))\n",
    "key_metrics(clean)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 5a - Line chart: revenue over time\n",
    "\n",
    "A line chart is the right choice when the horizontal axis is **time**. The dashed line is a 3-month moving average, which smooths out the ups and downs so the trend is easier to see."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "monthly = revenue_by_month(clean)\n",
    "plot_monthly_revenue(monthly)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 5b - Bar charts: comparing groups\n",
    "\n",
    "Bar charts compare **totals between groups**. The grouped version splits every category by region."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "by_category = summarize_by(clean, 'category')\n",
    "plot_bar(by_category['sum'], title='Revenue by category', ylabel='Revenue ($)')\n",
    "plt.show()\n",
    "\n",
    "plot_grouped_bar(revenue_by_category_region(clean), title='Revenue by category and region')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 5c - Histograms: the shape of the data\n",
    "\n",
    "A histogram shows **how the values of one column are spread out**. The dashed line is the mean and the dotted line is the median."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "plot_histogram(clean['customer_age'], title='Customer age', xlabel='Age (years)')\n",
    "plt.show()\n",
    "\n",
    "plot_histogram_grid(clean, ['customer_age', 'unit_price', 'revenue', 'customer_rating'])\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 5d - Correlation heatmap\n",
    "\n",
    "The heatmap shows how strongly every pair of numeric columns moves together, from -1 to +1. Remember: **correlation is not causation**, and some relationships are built in. Revenue is calculated from units, price and discount, so those columns are bound to correlate with it."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "corr = correlation_matrix(clean)\n",
    "plot_correlation_heatmap(corr)\n",
    "plt.show()\n",
    "\n",
    "strongest_correlations(corr, n=5)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 6 - Export the results\n",
    "\n",
    "Save the clean data and some tables so you can open them in Excel or share them. Everything goes into the `outputs/notebook` folder (it is ignored by git)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "out = config.OUTPUT_DIR / 'notebook'\n",
    "export_clean_data(clean, out / 'clean_sales.csv')\n",
    "export_tables({'monthly_revenue': monthly.to_frame(), 'correlations': corr}, out / 'tables')\n",
    "print('Saved to', out)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Try it yourself\n",
    "\n",
    "1. Change the moving average: `plot_monthly_revenue(monthly, rolling_window=6)`.\n",
    "2. Show only the top 3 categories: `plot_bar(by_category['sum'], top_n=3)`.\n",
    "3. Make a histogram of `marketing_spend`.\n",
    "4. Use Spearman instead of Pearson: `correlation_matrix(clean, method='spearman')`. What changes?\n",
    "5. Run the whole analysis from a terminal with `python -m dataviz.pipeline` and open `outputs/report.md`.\n",
    "6. Analyse your own file: it needs the same columns as `data/raw/sales_raw.csv`.\n",
    "\n",
    "Read `docs/analysis.md` for a deeper explanation of every step and how to read each chart."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}